In [1]:
import pandas as pd
import numpy as np
import os

# 1. Put Scrapped data into the dataset (1_Scrapped+Datasets)

### Adding new columns (DO NOT RERUN)

In [56]:
import os
import numpy as np
year=1999
i=0

while i<=25:
    data=pd.read_csv(os.path.join("Scrapped+Datasets","scimagojr "+str(year+i)+".csv"), sep=";")
    data.insert(11, "Est. value (USD) ("+str(year+i)+")", np.nan)
    data.insert(13, "Self-Cites (3years)", np.nan)
    data.insert(15, "Cited Docs. (3years)", np.nan)
    data.insert(16, "Uncited Docs. (3years)", np.nan)

    #data.head()
    data.to_csv(os.path.join("Scrapped+Datasets","scimagojr "+str(year+i)+".csv"),index=False)
    i+=1

In [1]:
import pandas as pd
import json
import os
import numpy as np

In [2]:
import ast
with open('All journals.txt','r') as f:
   All_journals = ast.literal_eval(f.read())
All_journals=list(All_journals)

- Self-cites - Evolution of the total number of citations and journal's self-citations received by a journal's published documents during the three previous years.
Journal Self-citation is defined as the number of citation from a journal citing article to articles published by the same journal.
- Cited (Uncited) doc - Ratio of a journal's items, grouped in three years windows, that have been cited at least once vs. those not cited during the following year.
- Est. value (USD) - It represents the potential financial worth of a journal. It is obtained by multiplying the journal's Estimated APC by the total number of citable documents published over the past five years. This value reflects the hypothetical revenue a journal could generate based on its estimated publication costs and scholarly output.

In [16]:
data=pd.read_csv(os.path.join("Scrapped+Datasets","scimagojr 2024.csv"))

In [21]:
sum(data["Sourceid"]=="123")

0

In [83]:
data.loc[data[data['Sourceid'] == All_journals[0]].index[0], ["Type"]]="kkk"

In [8]:

import time
start=time.time()

#TODO: for each year
for year in range(1999,2024):

    csvpath=os.path.join("Scrapped+Datasets", "scimagojr "+str(year)+".csv")
    
    with open(csvpath, "r") as file:
        #read this file:
        df=pd.read_csv(file)
        df["Est. value (USD) ("+str(year)+")"]=df["Est. value (USD) ("+str(year)+")"].astype("Int64")
        df["Self-Cites (3years)"]=df["Self-Cites (3years)"].astype("Int64")
        df["Cited Docs. (3years)"]=df["Cited Docs. (3years)"].astype("Int64")
        df["Uncited Docs. (3years)"]=df["Uncited Docs. (3years)"].astype("Int64")
        lookup_columns={'Est. value (USD)': "Est. value (USD) ("+str(year)+")", 'Self Cites':"Self-Cites (3years)", 'Cited documents':"Cited Docs. (3years)", 'Uncited documents': "Uncited Docs. (3years)"}
        #access value with sourceid
        
        
    
    # Open and read the JSON file
        for All_jour in All_journals:
            #check whether a journal is there:
            if sum(df["Sourceid"]==All_jour)==0:
                continue
            
            with open(os.path.join("All Scrapped Data",str(All_jour)+'.json'), 'r') as js:
                data = json.load(js)
                
                for category in ['Self Cites', 'Cited documents', 'Uncited documents', 'Est. value (USD)']:
                    years_js=data[category]["Year"]
                    values=data[category][category]
                    for j in range(len(years_js)):
                        year_js=int(years_js[j])
                        value=values[j]
                        if year_js==year:
                        #put this value in the appropriate place in csv:
                            if value=="":
                                value=np.nan
                            else:
                                value=int(value)
                            df.loc[df[df['Sourceid'] == All_jour].index[0], [lookup_columns[category]]]=value
        
        df.to_csv(os.path.join("Scrapped+Datasets", "DONE_scimagojr "+str(year)+".csv"), index=False)          
                

end=time.time()
print(end-start)

8826.744332313538


In [5]:
#iterate over 40k files

import time
start=time.time()
for All_jour in All_journals:
    with open(os.path.join("All Scrapped Data",str(All_jour)+'.json'), 'r') as file:
        dat = json.load(file)
        adat['Self Cites']
end=time.time()
print(end-start)


12.770341873168945


In [ ]:
# ONLY JOURNALS ALLOWED:
for year in range(1999,2025):
    csvpath=os.path.join("Scrapped+Datasets", "DONE_scimagojr "+str(year)+".csv")
    df=pd.read_csv(csvpath, encoding="utf-8")
    df = df[df.Type == "journal"]
    df.to_csv(os.path.join("Scrapped+Datasets", "1DONE_scimagojr "+str(year)+".csv"), index=False) 
    #os.rename(os.path.join("Scrapped+Datasets", "1DONE_scimagojr "+str(year)+".csv"),os.path.join("Scrapped+Datasets", "DONE_scimagojr "+str(year)+".csv"))

# 2. Preparing Final Datasets (Prepared Datasets folder)

In [66]:
for year in range(2024,2025):
    
    csvpath=os.path.join("Scrapped+Datasets", "DONE_scimagojr "+str(year)+".csv")
    df=pd.read_csv(csvpath, encoding="utf-8")
    #pick certain columns:
    
    attributes_keep=['Title', 'SJR', 'H index', "Total Docs. ("+str(year)+")",'Total Docs. (3years)', 'Total Refs.',
       'Est. value (USD) ('+str(year)+')', 'Total Cites (3years)', 'Self-Cites (3years)', 'Citable Docs. (3years)','Uncited Docs. (3years)', 
                     'Cites / Doc. (2years)', 'Ref. / Doc.','Publisher', 'Coverage']
    #creating new features, deleting old
    df=df[attributes_keep]
    temp=df["Self-Cites (3years)"]/df["Total Cites (3years)"]
    df.insert(8,"Self-Cites/Total Cites (3years)", temp)
    df=df.drop(columns="Self-Cites (3years)")

    temp=df["Uncited Docs. (3years)"]/df["Total Docs. (3years)"]
    df.insert(9,"Uncited Docs./Total Docs. (3years)", temp)
    df=df.drop(columns="Uncited Docs. (3years)")

    #creating Coverage_Duration
    df.Coverage=df.Coverage.astype("str")
    df.insert(13,"Coverage_Duration",0)
    for i in range(len(df.index)):
        count = 0
        years=df["Coverage"][i].split(", ")
        for j in years:
            years1=j.split('-')
            if len(years1) == 1:
                count += 1
            else:
                count += int(years1[1]) - int(years1[0]) +1
        df.loc[i,"Coverage_Duration"]=count
    df=df.drop(columns="Coverage")  

    
    df.to_csv(os.path.join("Prepared Datasets", "Prepd_scimagojr "+str(year)+".csv"), index=False)  
 

## Tests

In [48]:
csvpath=os.path.join("Scrapped+Datasets", "DONE_scimagojr "+str(1999)+".csv")
year=1999
df=pd.read_csv(csvpath, encoding="utf-8")
attributes_keep=['Title', 'SJR', 'H index', "Total Docs. ("+str(year)+")",'Total Docs. (3years)', 'Total Refs.',
       'Est. value (USD) ('+str(year)+')', 'Total Citations (3years)', 'Self-Cites (3years)', 'Citable Docs. (3years)','Uncited Docs. (3years)', 'Citations / Doc. (2years)', 'Ref. / Doc.','Publisher',
                'Coverage']
df=df[attributes_keep]

#changing them:#


In [49]:
temp=df["Self-Cites (3years)"]/df["Total Citations (3years)"]
df.insert(8,"Self-Cites/Total Cites (3years)", temp)
df=df.drop(columns="Self-Cites (3years)")

In [50]:
temp=df["Uncited Docs. (3years)"]/df["Total Docs. (3years)"]
df.insert(9,"Uncited Docs./Total Docs. (3years)", temp)
df=df.drop(columns="Uncited Docs. (3years)")

In [58]:
df.Cooverage=df.Coverage.astype("str")

0                   1974-2025
1                   1978-2024
2                   1987-2025
3                   1994-2025
4                   1989-2025
                 ...         
15628               1996-2000
15629               1996-2024
15630    1979-1981, 1991-2001
15631               2022-2025
15632               1996-2025
Name: Coverage, Length: 15633, dtype: object

In [60]:
df.Coverage=df.Coverage.astype("str")
df.insert(13,"Coverage_Duration",0)
for i in range(len(df.index)):
    count = 0
    years=df["Coverage"][i].split(", ")
    for j in years:
        years1=j.split('-')
        if len(years1) == 1:
            count += 1
        else:
            count += int(years1[1]) - int(years1[0]) +1
    df.loc[i,"Coverage_Duration"]=count

In [54]:
len(df.index)


15633

In [61]:
df

,Title,SJR,H index,Total Docs. (1999),Total Docs. (3years),Total Refs.,Est. value (USD) (1999),Self-Cites/Total Cites (3years),Total Citations (3years),Uncited Docs./Total Docs. (3years),Citable Docs. (3years),Citations / Doc. (2years),Ref. / Doc.,Publisher,Coverage_Duration,Coverage
0,Cell,"43,449",925,351,1340,15964,2342559.0,0.019236,48295,0.004478,1332,"34,68","45,48",Elsevier B.V.,52,1974-2025
1,Annual Review of Neuroscience,"25,760",269,21,60,3419,127273.0,0.011029,1632,0.000000,60,"22,79","162,81",Annual Reviews Inc.,47,1978-2024
2,Genes and Development,"25,272",489,298,889,16623,1799712.0,0.037683,17382,0.004499,888,"18,97","55,78",Cold Spring Harbor Laboratory Press,39,1987-2025
3,Immunity,"22,298",475,151,438,7770,891122.0,0.044025,9222,0.000000,438,"20,47","51,46",Cell Press,32,1994-2025
4,Current Opinion in Cell Biology,"21,691",284,104,318,5829,610637.0,0.013327,6603,0.110063,298,"22,57","56,05",Elsevier Ltd,37,1989-2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15628,Zeitschrift fur Onkologie,NaN,7,20,58,505,NaN,0.000000,6,0.948276,53,"0,02","25,25",Karl F. Haug Verlag in MVS Medizinverlage Stut...,5,1996-2000
15629,Zeitschrift fur Unternehmensgeschichte,NaN,10,12,24,335,NaN,0.000000,1,0.958333,22,"0,00","27,92",De Gruyter Oldenbourg,29,1996-2024
15630,Zeitschrift fur Vermessungswesen,NaN,10,44,163,544,NaN,0.600000,30,0.846626,163,"0,21","12,36",Wissner Verlag,14,"1979-1981, 1991-2001"
15631,ZFW - Advances in Economic Geography,NaN,23,17,44,745,NaN,0.217391,23,0.659091,44,"0,21","43,82",Walter de Gruyter GmbH,4,2022-2025


# 3. Cleaning Final Dataset (Prepared Datasets folder; CLEAN_Prepd_scimagojr file)
> After this step the Data is fully ready for ML

1. Nan get rid of
2. the duplicates of the same journals names were deleted
3. format all the data points
4. prepare the Right data types

In [69]:
csvpath=os.path.join("Prepared Datasets", "Prepd_scimagojr "+str(1999)+".csv")

df=pd.read_csv(csvpath, encoding="utf-8")

## 1. Nan values: (delete - about 20% ; 80%left after deletion)

SJR      
Est. value (USD) (1999)                 
Self-Cites/Total Cites (3years)         
Uncited Docs./Total Docs. (3years)      
Publisher                                


In [117]:
df.dropna(inplace=True, ignore_index=True)
df.drop_duplicates(subset=["Title"], keep="first",ignore_index=True, inplace=True)

In [120]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11921 entries, 0 to 11920
Data columns (total 15 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Title                               11921 non-null  object 
 1   SJR                                 11921 non-null  float64
 2   H index                             11921 non-null  int64  
 3   Total Docs. (1999)                  11921 non-null  int64  
 4   Total Docs. (3years)                11921 non-null  int64  
 5   Total Refs.                         11921 non-null  int64  
 6   Est. value (USD) (1999)             11921 non-null  float64
 7   Total Cites (3years)                11921 non-null  int64  
 8   Self-Cites/Total Cites (3years)     11921 non-null  float64
 9   Uncited Docs./Total Docs. (3years)  11921 non-null  float64
 10  Citable Docs. (3years)              11921 non-null  int64  
 11  Cites / Doc. (2years)               11921

In [103]:
df.dropna(inplace=True, ignore_index=True)

## 2. the duplicates of the same journals names deleted (left only with the better SJR)

In [115]:
df.drop_duplicates(subset=["Title"], keep="first",ignore_index=True, inplace=True)

## 3-4. Format Float + prepare the Right Data types

In [119]:
#format FLOAT values
for i in range(df.shape[0]):
    df.loc[i, "Cites / Doc. (2years)"] = df.loc[i, "Cites / Doc. (2years)"].replace(",", ".")
    df.loc[i, "Ref. / Doc."] = df.loc[i, "Ref. / Doc."].replace(",", ".")
    df.loc[i, "SJR"] = df.loc[i, "SJR"].replace(",", ".")
df["Cites / Doc. (2years)"]=df["Cites / Doc. (2years)"].astype(np.float64)
df["Ref. / Doc."]=df["Ref. / Doc."].astype(np.float64)
df["SJR"]=df["SJR"].astype(np.float64)

In [75]:
df.dtypes

Title                                  object
SJR                                    object
H index                                 int64
Total Docs. (2024)                      int64
Total Docs. (3years)                    int64
Total Refs.                             int64
Est. value (USD) (2024)               float64
Total Cites (3years)                    int64
Self-Cites/Total Cites (3years)       float64
Uncited Docs./Total Docs. (3years)    float64
Citable Docs. (3years)                  int64
Cites / Doc. (2years)                  object
Ref. / Doc.                            object
Coverage_Duration                       int64
Publisher                              object
dtype: object

In [68]:
#df.reset_index(drop=True, inplace=True)

,Title,SJR,H index,Total Docs. (1999),Total Docs. (3years),Total Refs.,Est. value (USD) (1999),Total Cites (3years),Self-Cites/Total Cites (3years),Uncited Docs./Total Docs. (3years),Citable Docs. (3years),Cites / Doc. (2years),Ref. / Doc.,Coverage_Duration,Publisher
0,Cell,"43,449",925,351,1340,15964,2342559.0,48295,0.019236,0.004478,1332,"34,68","45,48",52,Elsevier B.V.
1,Annual Review of Neuroscience,"25,760",269,21,60,3419,127273.0,1632,0.011029,0.000000,60,"22,79","162,81",47,Annual Reviews Inc.
2,Genes and Development,"25,272",489,298,889,16623,1799712.0,17382,0.037683,0.004499,888,"18,97","55,78",39,Cold Spring Harbor Laboratory Press
3,Immunity,"22,298",475,151,438,7770,891122.0,9222,0.044025,0.000000,438,"20,47","51,46",32,Cell Press
4,Current Opinion in Cell Biology,"21,691",284,104,318,5829,610637.0,6603,0.013327,0.110063,298,"22,57","56,05",37,Elsevier Ltd
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15628,Zeitschrift fur Onkologie,NaN,7,20,58,505,NaN,6,0.000000,0.948276,53,"0,02","25,25",5,Karl F. Haug Verlag in MVS Medizinverlage Stut...
15629,Zeitschrift fur Unternehmensgeschichte,NaN,10,12,24,335,NaN,1,0.000000,0.958333,22,"0,00","27,92",29,De Gruyter Oldenbourg
15630,Zeitschrift fur Vermessungswesen,NaN,10,44,163,544,NaN,30,0.600000,0.846626,163,"0,21","12,36",14,Wissner Verlag
15631,ZFW - Advances in Economic Geography,NaN,23,17,44,745,NaN,23,0.217391,0.659091,44,"0,21","43,82",4,Walter de Gruyter GmbH


## Results:

In [121]:
for year in range(1999,2025):
    csvpath=os.path.join("Prepared Datasets", "Prepd_scimagojr "+str(year)+".csv")

    df=pd.read_csv(csvpath, encoding="utf-8")
    df.dropna(inplace=True, ignore_index=True)
    df.drop_duplicates(subset=["Title"], keep="first",ignore_index=True, inplace=True)
    #format FLOAT values
    for i in range(df.shape[0]):
        df.loc[i, "Cites / Doc. (2years)"] = df.loc[i, "Cites / Doc. (2years)"].replace(",", ".")
        df.loc[i, "Ref. / Doc."] = df.loc[i, "Ref. / Doc."].replace(",", ".")
        df.loc[i, "SJR"] = df.loc[i, "SJR"].replace(",", ".")
    df["Cites / Doc. (2years)"]=df["Cites / Doc. (2years)"].astype(np.float64)
    df["Ref. / Doc."]=df["Ref. / Doc."].astype(np.float64)
    df["SJR"]=df["SJR"].astype(np.float64)
    df.to_csv(os.path.join("Prepared Datasets", "CLEAN_Prepd_scimagojr "+str(year)+".csv"), index=False)